In [1]:
"""
02 - VALIDATE PCA COMPONENT COUNTS
Fits no model here. Loads pca_model.npz trained on training regions,
then evaluates unseen validation patches.
"""

import os
import io
import json
import numpy as np
import pandas as pd
from skimage.metrics import structural_similarity

BASE = "/kaggle/input/datasets/krrishshandilya/dipcoma-trainval"
DATASET = os.path.join(BASE, "dataset")
RESULTS = "/kaggle/working/pca_results"
MODEL_PATH = "/kaggle/input/notebooks/arvindreddy99920/pca-model/pca_results/pca_model.npz"
os.makedirs(RESULTS, exist_ok=True)

COMPONENT_COUNTS = [3, 4, 5, 6, 7, 8, 9, 10, 11, 13]

def slugify(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")

model = np.load(MODEL_PATH)
mean = model["mean"].astype(np.float32)
components = model["components"].astype(np.float32)
LO = model["LO"].astype(np.float32)
HI = model["HI"].astype(np.float32)

patch_log = pd.read_csv(os.path.join(BASE, "patch_log.csv"))
val_rows = patch_log[patch_log["split"].str.lower() == "val"].copy()

details = []

for patch_number, (_, row) in enumerate(val_rows.iterrows(), start=1):
    patch_path = os.path.join(
        DATASET, "val", slugify(row["region"]), row["patch"]
    )
    if not os.path.isfile(patch_path):
        raise FileNotFoundError(patch_path)

    raw = np.load(patch_path).astype(np.float32)
    original = (
        np.clip(raw, LO[:, None, None], HI[:, None, None])
        - LO[:, None, None]
    ) / (HI[:, None, None] - LO[:, None, None])

    X = original.reshape(13, -1).T
    centered = X - mean
    original_bytes = os.path.getsize(patch_path)

    for k in COMPONENT_COUNTS:
        selected = components[:k]
        scores = centered @ selected.T

        score_min = scores.min(axis=0)
        score_max = scores.max(axis=0)
        score_range = np.maximum(score_max - score_min, 1e-8)

        quantized = np.round(
            (scores - score_min) / score_range * 255
        ).clip(0, 255).astype(np.uint8)

        restored_scores = (
            quantized.astype(np.float32) / 255.0
        ) * score_range + score_min

        reconstructed = restored_scores @ selected + mean
        reconstructed = reconstructed.T.reshape(13, 512, 512)

        mse = np.mean((original - reconstructed) ** 2)
        psnr = 10.0 * np.log10(1.0 / max(mse, 1e-15))

        band_ssim = [
            structural_similarity(
                original[b], reconstructed[b], data_range=1.0
            )
            for b in range(13)
        ]
        mean_ssim = float(np.mean(band_ssim))

        # Measure real .npz bytes, including quantization min/max metadata.
        buffer = io.BytesIO()
        np.savez_compressed(
            buffer,
            scores=quantized,
            score_min=score_min,
            score_max=score_max,
        )
        compressed_bytes = buffer.tell()
        ratio = original_bytes / compressed_bytes

        details.append({
            "patch": row["patch"],
            "region": row["region"],
            "components": k,
            "PSNR": psnr,
            "SSIM": mean_ssim,
            "original_bytes": original_bytes,
            "compressed_bytes": compressed_bytes,
            "compression_ratio": ratio,
            "storage_reduction": (1.0 - 1.0 / ratio) * 100.0,
        })

    print(f"Evaluated validation patch {patch_number}/{len(val_rows)}")

details_df = pd.DataFrame(details)
details_path = os.path.join(RESULTS, "validation_details.csv")
details_df.to_csv(details_path, index=False)

summary_df = (
    details_df.groupby("components", as_index=False)
    .agg({
        "PSNR": "mean",
        "SSIM": "mean",
        "compression_ratio": "mean",
        "storage_reduction": "mean",
        "compressed_bytes": "mean",
    })
)
summary_path = os.path.join(RESULTS, "validation_summary.csv")
summary_df.to_csv(summary_path, index=False)

region_df = (
    details_df.groupby(["region", "components"], as_index=False)
    .agg({
        "PSNR": "mean",
        "SSIM": "mean",
        "compression_ratio": "mean",
        "storage_reduction": "mean",
    })
)
region_path = os.path.join(RESULTS, "validation_region_summary.csv")
region_df.to_csv(region_path, index=False)

print("\nOVERALL VALIDATION SUMMARY")
print(summary_df.round(4).to_string(index=False))
print("\nSaved:")
print(details_path)
print(summary_path)
print(region_path)


Evaluated validation patch 1/165
Evaluated validation patch 2/165
Evaluated validation patch 3/165
Evaluated validation patch 4/165
Evaluated validation patch 5/165
Evaluated validation patch 6/165
Evaluated validation patch 7/165
Evaluated validation patch 8/165
Evaluated validation patch 9/165
Evaluated validation patch 10/165
Evaluated validation patch 11/165
Evaluated validation patch 12/165
Evaluated validation patch 13/165
Evaluated validation patch 14/165
Evaluated validation patch 15/165
Evaluated validation patch 16/165
Evaluated validation patch 17/165
Evaluated validation patch 18/165
Evaluated validation patch 19/165
Evaluated validation patch 20/165
Evaluated validation patch 21/165
Evaluated validation patch 22/165
Evaluated validation patch 23/165
Evaluated validation patch 24/165
Evaluated validation patch 25/165
Evaluated validation patch 26/165
Evaluated validation patch 27/165
Evaluated validation patch 28/165
Evaluated validation patch 29/165
Evaluated validation pa